# LLM Training Usage and Cost Estimates for 1B/3B/8B/70B Models

Calculate the floating point operations for 1B/3B/8B/70B models.

## Technical Details: The Math Behind It
We used two main ways to calculate the computational effort (FLOPs - Floating Point Operations):

### 1. The "FLOPS" Approximation
This is a standard rule of thumb for estimating training cost.
- **Formula:** `FLOPs = 6 * Number_of_Params * Number_of_Tokens`
- **Why 6?** The factor of 6 accounts for the total operations in one training step (Forward + Backward):
  - **Forward Pass:** `2 * N * D` (1 multiply + 1 accumulate per parameter).
  - **Backward Pass:** `4 * N * D` (Calculating gradients for weights and input).
  - **Total:** `2 (Forward) + 4 (Backward) = 6` FLOPs per parameter per token.

### 2. Attention-Aware Formula (More Precise)
For a single micro batch, we calculated:
`FLOPS = (6 * seq_len * num_params) + (12 * num_layers * hidden_size * seq_len^2)`
- This includes the "attention mechanism" calculations which grow quadratically with sequence length.

## Configurations & Results
We ran two types of estimates: one for the total training duration and another for the computational cost of a single training step.

### 1. Training Duration Estimates
Parameters used:
- **Hardware:** 8x NVIDIA H100 GPUs
- **Utilization:** 30% MFU

| Model Size | Training Tokens | Estimated Time |
| :--- | :--- | :--- |
| **1B** Parameters | 20 Billion | ~0.70 days |
| **3B** Parameters | 40 Billion | ~4.17 days |
| **8B** Parameters | 100 Billion | ~27.82 days |
| **70B** Parameters | 240 Billion | ~584.27 days |

### 2. Per-Step Compute Intensity (PFLOPs)
Parameters used:
- **Sequence Length:** 2048
- **Micro Batch Size:** 32

| Model Size | PFLOPs per Step |
| :--- | :--- |
| **1B** | **0.39** |
| **3B** | **1.18** |
| **8B** | **3.15** |
| **70B** | **27.53** |

## Hardware Assumptions
- **GPU:** NVIDIA H100
- **Peak Performance:** 832 TFLOPS (Tera-FLOPS) per GPU (FP16).
- **Utilization (MFU):** We assumed we can only use **30% (0.3)** of the theoretical peak speed due to communication overhead.
- **System:** 8 GPUs working together.

## TO DO List
We are planning to expand this work with the following items:

- [ ] **Scale to 16 GPUs:** Update calculations for a larger cluster.
- [ ] **Memory & Sharding Analysis:**
  - Calculate max memory usage with **Zero2** vs **Zero3** sharding.
  - Investigate memory requirements for each sharding technique.
- [ ] **Create Configuration Files (YAML):**
  - Create 4 separate config files for **1B, 3B, 8B, and 70B** models.
  - specific open-source details: Sequence length, Number of Experts (MoE), Attention Heads.
  - For each config:
    1. Calculate total FLOPs.
    2. Calculate total memory required.
- [ ] **Hardware Comparison (H100 vs Blackwell):**
  - Compare Total VRAM per machine.
  - Compare Max TFLOPS achievable for each datatype.
- [ ] **Detailed Resource Planning:**
  - **Sequence Length & Batch Size:** Analyze impact on memory.
  - **Memory per GPU:**
    - Estimate specific holdings (e.g., 1B model → ~16GB data).
    - Estimate for 70B model (scaled).
  - **Trade-offs:** Gradient Accumulation vs. Activation Checkpointing (Memory vs. Compute).
  - **Starting Loss:** Estimate starting loss for 1B tokens (1 epoch).
- [ ] **Final Output Estimations:**
  - Map Config → Machine (FLOPs per datatype), Model and Optimization strategies.
  - Total hours required to train.
  - Max memory peak observed per machine.

In [1]:
# Model / training config
num_params   = 1_000_000_000      # 1B parameters
num_layers   = 24                 # example
h            = 2048               # hidden size
seq_len      = 2048               # context length
micro_bs     = 4                  # micro-batch size


In [2]:
# Tokens processed per step
num_tokens = seq_len * micro_bs
num_tokens

8192

forward pass

In [3]:
flops_forward = 2 * num_tokens * num_params
flops_forward/1e15

0.016384

Backward pass

In [4]:
flops_backward = 4 * num_tokens * num_params
flops_backward/1e15

0.032768

Total

In [5]:
flops_total_simple = flops_forward + flops_backward
flops_total_simple/1e15

0.049152

FLOPs formula (attention-aware)

In [6]:
# Per single sample (1 batch)
flops_per_sample = (
    6 * seq_len * num_params
    + 12 * num_layers * h * (seq_len ** 2)
)
flops_per_sample/1e15

0.014761901162496

In [7]:
# For a micro-batch
flops_total = micro_bs * flops_per_sample
flops_total/1e15

0.059047604649984

when can you ignore attention FLOPs?

In [9]:
attention_flops = 12 * num_layers * h * (seq_len ** 2)
param_flops     = 6 * seq_len * num_params  ### this is just for 1 batch, we need to compute 20B tokens 6 * 20B*1B

print("Attention / Param FLOPs ratio:", attention_flops / param_flops)

Attention / Param FLOPs ratio: 0.201326592


~20% of your total FLOPs are coming from attention, and ~80% from parameter (dense matmul) compute.

| Component                       | Share of FLOPs |
| ------------------------------- | -------------- |
| Dense matmuls (QKV, MLP, grads) | ~80%           |
| Attention (QKᵀ, softmax, AV)    | ~20%           |


In [10]:
def training_flops(
    num_params: float,
    seq_len: int,
    batch_size: int,
    flops_per_token_param: int = 6,
):
    """
    Estimate FLOPs for one training step (forward + backward).

    Args:
        num_params (float): Number of model parameters (e.g. 1e9, 70e9)
        seq_len (int): Sequence length
        batch_size (int): Global batch size
        flops_per_token_param (int): Multiplier (default=6 for fwd+bwd)
    """
    tokens_per_step = seq_len * batch_size
    flops = flops_per_token_param * tokens_per_step * num_params

    return {
        "tokens_per_step": tokens_per_step,
        "flops_per_step": flops,
        "peta_flops_per_step": flops / 1e15,
    }


number of tokens

In [11]:
models = {
    "1B": 1e9,
    "3B": 3e9,
    "8B": 8e9,
    "70B": 70e9,
}

for name, params in models.items():
    out = training_flops(
        num_params=params,
        seq_len=2048,
        batch_size=32
    )
    print(f"{name}: {out['peta_flops_per_step']:.2f} PFLOPs/step")


1B: 0.39 PFLOPs/step
3B: 1.18 PFLOPs/step
8B: 3.15 PFLOPs/step
70B: 27.53 PFLOPs/step


A100 GPU

In [12]:
# Finally let's check out the 6ND approximation as total cost of training in FLOPs
N = 1e9  # this is number of parameters, N
D = 20e9  # 300B tokens, this is dataset size in tokens, D
a = 312e12  # 312 TFLOPS
assumed_mfu = 0.3  # assume this model flops utilization (take the current 30% from above and add some DDP overhead)
flops_throughput = a100_bfloat16_promised_flops * 8 * assumed_mfu  # assume an 8XA100 node at 30% utilization
flops_needed = 6 * N * D
time_needed_over_all_tokens_in_seconds = flops_needed / flops_throughput  # in seconds
print(f"time needed to train the model: {time_needed_over_all_tokens_in_seconds/3600/24:.2f} days")100_bfloat16_promised_flops

time needed to train the model: 1.85 days


H100 GPU

In [13]:
def training_time_h100(
    num_params: float,
    total_tokens: float,
    gpu_peak_flops: float,
    num_gpus: int = 1,
    mfu: float = 0.3,
):
    """
    Estimate total training time in seconds/days using the 6ND approximation.

    Args:
        num_params (float): Total model parameters (N)
        total_tokens (float): Total dataset tokens (D)
        gpu_peak_flops (float): Single GPU peak FLOPS (in FLOPs, e.g., 832e12 for H100 FP16)
        num_gpus (int): Number of GPUs
        mfu (float): Model flops utilization (0-1)

    Returns:
        dict with total_flops, flops_per_second, time_seconds, time_days
    """
    # Total FLOPs using 6*N*D approximation
    flops_needed = 6 * num_params * total_tokens

    # Effective FLOPS throughput across all GPUs
    flops_throughput = gpu_peak_flops * num_gpus * mfu

    # Training time in seconds
    time_seconds = flops_needed / flops_throughput
    time_days = time_seconds / (3600 * 24)
    # time_hours
    return {
        "total_flops": flops_needed,
        "flops_per_second": flops_throughput,
        "time_seconds": time_seconds,
        "time_days": time_days
    }


H100 example

In [16]:
# H100 peak FP16 ~ 832 TFLOPS per GPU
h100_peak_flops = 832e12  # FLOPS

stages = {
    "Stage 1 - 1B Dense": {"N": 1e9, "D": 20e9},
    "Stage 2 - 3B MoE-small": {"N": 3e9, "D": 40e9},
    "Stage 3 - 8B Dense-deep": {"N": 8e9, "D": 100e9},
    "Stage 4 - 70B MoE-large": {"N": 70e9, "D": 240e9},
}

num_gpus = 8
mfu = 0.3  # assuming 30% utilization

for stage, vals in stages.items():
    res = training_time_h100(
        num_params=vals["N"],
        total_tokens=vals["D"],
        gpu_peak_flops=h100_peak_flops,
        num_gpus=num_gpus,
        mfu=mfu
    )
    print(f"{stage}: {res['time_days']:.2f} days")


Stage 1 - 1B Dense: 0.70 days
Stage 2 - 3B MoE-small: 4.17 days
Stage 3 - 8B Dense-deep: 27.82 days
Stage 4 - 70B MoE-large: 584.27 days
